In [1]:
# ================================================================
# NOTEBOOK : nb_silver_weather
# Read from  : bronze_lakehouse → bronze_weather
# Write to   : silver_lakehouse → silver_weather
# ================================================================

StatementMeta(, 1f09c3e3-6525-4d5e-a4b4-d93c30e00ced, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F 
from pyspark.sql.functions import trim, initcap, col, to_date, to_timestamp

bronze_df = spark.sql("SELECT * FROM bronze_lakehouse.bronze_weather")
print(f"[BRONZE] Rows:{bronze_df.count()}")
display(bronze_df)

StatementMeta(, 1f09c3e3-6525-4d5e-a4b4-d93c30e00ced, 4, Finished, Available, Finished, False)

[BRONZE] Rows:40


SynapseWidget(Synapse.DataFrame, 854c5345-2049-4431-b0b3-fb64b8dab2fb)

In [6]:

from pyspark.sql import functions as F

silver_df = ( bronze_df
# ── Nulls + Dedup ──────────────────────────────────────────
.filter(col("forecast_time").isNotNull())
.filter(col("City").isNotNull())
.drop_duplicates(["City", "forecast_time"])

# ── Fix data types ─────────────────────────────────────────
.withColumn("forecast_time",    to_timestamp(col("forecast_time")))
.withColumn("forecast_date",   to_date(col("forecast_time")))
.withColumn("temp_c",   col("temp_c").cast("float"))
.withColumn("humidity_pct",    col("humidity_pct").cast("integer"))
.withColumn("pressure_hpa",    col("pressure_hpa").cast("integer"))
.withColumn("wind_speed_ms",    col("wind_speed_ms").cast("float"))
.withColumn("cloudiness_pct",  col("cloudiness_pct").cast("integer"))
.withColumn("rain_3h_mm",         col("rain_3h_mm").cast("float"))

# ── Standardise ────────────────────────────────────────────
.withColumn("City",     trim(initcap("City")))
.withColumn("weather_main",     trim(initcap("weather_main")))

# ── Derived columns ────────────────────────────────────────
.withColumn("is_rainy",    F.col("weather_main").isin(["Rain", "Drizzle", "Thunderstorm"]))



.withColumn("temp_category",    F.when(col("temp_c") < 20, "Cool")
    .when(F.col("temp_c") < 30, "Warm")
    .otherwise("Hot"))

.withColumn("humidity_category", F.when(col("humidity_pct") < 40, "Low" )
    .when(F.col("humidity_pct") < 70, "Moderate")
    .otherwise("High"))

# Hour of day — useful for joining with intraday sales
.withColumn("forecast_hour", F.hour("forecast_time"))


# ── Metadata ───────────────────────────────────────────────
.withColumn("_silver_load_ts", F.current_timestamp())
.drop("country")   # not needed beyond bronze
)

silver_df.write.format("delta").mode("overwrite")\
.option("overwriteSchema", "true").saveAsTable("silver_weather")


print(f"[DONE] silver_weather written:{silver_df.count()} rows")
display(silver_df)



StatementMeta(, 1f09c3e3-6525-4d5e-a4b4-d93c30e00ced, 8, Finished, Available, Finished, False)

[DONE] silver_weather written:40 rows


SynapseWidget(Synapse.DataFrame, 7ca606d6-b92a-4173-8c0a-cdca8942af11)